# Preparar datos para consumir
se toman datos de resultados (csv) del año 2010 al actual (2026) , se ordenan y se tratan para obtener los promedios de goles de los 5 ultimos partidos por equipo en cada partido en el registro.

luego se crean las demás variables.

In [9]:
import pandas as pd
from datetime import datetime

In [10]:
hoy = datetime.now()
df = pd.read_csv("../data/results.csv", parse_dates=["date"])
# matches historical data from 2010 to 2026
df = df[df["date"] > "2010-01-01"]
df = df.dropna(subset=["home_score"])
# print(f" max date: {df['date'].max()}")
# print(f" min date: {df['date'].min()}")
# ordenar por fecha
# df = df.sort_values("date").reset_index(drop=True)

# diccionario para guardar historial por equipo
team_history = {}

# listas donde guardaremos resultados
home_avg_scored = []
home_avg_conceded = []
away_avg_scored = []
away_avg_conceded = []

# función para calcular promedios
def get_avg_last_5(matches):
    last_5 = matches[-5:]
    
    if len(last_5) == 0:
        return 0, 0
    
    goals_scored = [m[0] for m in last_5]
    goals_conceded = [m[1] for m in last_5]
    
    return sum(goals_scored)/len(goals_scored), sum(goals_conceded)/len(goals_conceded)

# recorrer partidos
for _, row in df.iterrows():
    
    home = row["home_team"]
    away = row["away_team"]
    
    home_score = row["home_score"]
    away_score = row["away_score"]
    
    # inicializar equipos si no existen
    if home not in team_history:
        team_history[home] = []
    if away not in team_history:
        team_history[away] = []
    
    # calcular promedios ANTES de guardar el partido actual
    h_scored, h_conceded = get_avg_last_5(team_history[home])
    a_scored, a_conceded = get_avg_last_5(team_history[away])
    
    home_avg_scored.append(h_scored)
    home_avg_conceded.append(h_conceded)
    away_avg_scored.append(a_scored)
    away_avg_conceded.append(a_conceded)
    
    # ahora sí guardar el partido actual en el historial
    team_history[home].append((home_score, away_score))
    team_history[away].append((away_score, home_score))

# agregar columnas al dataframe
df["home_avg_goals_scored_5"] = home_avg_scored
df["home_avg_goals_conceded_5"] = home_avg_conceded
df["away_avg_goals_scored_5"] = away_avg_scored
df["away_avg_goals_conceded_5"] = away_avg_conceded

In [ ]:
pesos = {
    "Friendly": 1,
    "FIFA World Cup": 3,
    "FIFA World Cup qualification": 2
}


df["peso"] = df["tournament"].map(pesos).fillna(1)

# recencia o que tanto vale un partido segun la fecha 
df["recencia"] = df["date"].apply(
    lambda x: 1 / (1 + (hoy - x).days / 365)
)
matches = df.drop(columns=["country","city","neutral"])
#matches

,date,home_team,away_team,home_score,away_score,tournament,home_avg_goals_scored_5,home_avg_goals_conceded_5,away_avg_goals_scored_5,away_avg_goals_conceded_5,peso,recencia
33583,2010-01-02,Iran,North Korea,1.0,0.0,Friendly,0.0,0.0,0.0,0.0,1.0,0.057826
33584,2010-01-02,Qatar,Mali,0.0,0.0,Friendly,0.0,0.0,0.0,0.0,1.0,0.057826
33585,2010-01-02,Syria,Zimbabwe,6.0,0.0,Friendly,0.0,0.0,0.0,0.0,1.0,0.057826
33586,2010-01-02,Yemen,Tajikistan,0.0,1.0,Friendly,0.0,0.0,0.0,0.0,1.0,0.057826
33587,2010-01-03,Angola,Gambia,1.0,1.0,Friendly,0.0,0.0,0.0,0.0,1.0,0.057836
...,...,...,...,...,...,...,...,...,...,...,...,...
49210,2026-03-31,Kosovo,Turkey,0.0,1.0,FIFA World Cup qualification,1.6,0.8,3.0,0.8,2.0,0.960526
49211,2026-03-31,Czech Republic,Denmark,2.0,2.0,FIFA World Cup qualification,2.0,0.8,3.4,1.4,2.0,0.960526
49212,2026-03-31,Cameroon,China PR,2.0,0.0,FIFA Series,1.0,1.2,1.0,1.0,1.0,0.960526
49213,2026-03-31,Australia,Curaçao,5.0,1.0,FIFA Series,0.6,1.2,2.0,0.6,1.0,0.960526


In [ ]:
# ranking fifa 2026
ranking = pd.read_csv("../data/ranking_fifa.csv",encoding="utf-8")
ranking

df = matches.merge(
    ranking,
    left_on = "home_team",
    right_on= "country",
    how="left"
)

df = df.rename(columns={"rank": "home_rank","points":"home_points"})
df =df.drop(columns=["country"])

df = df.merge(
    ranking,
    left_on = "away_team",
    right_on= "country",
    how="left"
)

df = df.rename(columns={"rank": "away_rank","points":"away_points"})
df =df.drop(columns=["country"])
df["home_level"]  = 1 / df["home_rank"].fillna(211)
df["away_level"]  = 1 / df["away_rank"].fillna(211)
df["resultado"] = (df["home_score"]>df["away_score"]).astype(int)
df["diff_rank"] = df["away_rank"] - df["home_rank"]
df["goal_diff"] = df["home_score"] - df["away_score"]
df["total_goal"] = df["home_score"] + df["away_score"]
df["level_diff"] = df["home_level"] - df["away_level"]
df =df.drop(columns=["home_rank","away_rank","home_points","away_points","tournament","date"])

df.to_csv("../data/clean_data.csv")

#df

,home_team,away_team,home_score,away_score,home_avg_goals_scored_5,home_avg_goals_conceded_5,away_avg_goals_scored_5,away_avg_goals_conceded_5,peso,recencia,home_level,away_level,resultado,diff_rank,goal_diff,total_goal,level_diff
0,Iran,North Korea,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.057826,0.004739,0.004739,1,NaN,1.0,1.0,0.000000
1,Qatar,Mali,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.057826,0.018182,0.019231,0,-3.0,0.0,0.0,-0.001049
2,Syria,Zimbabwe,6.0,0.0,0.0,0.0,0.0,0.0,1.0,0.057826,0.011905,0.007692,1,46.0,6.0,6.0,0.004212
3,Yemen,Tajikistan,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.057826,0.006711,0.009709,0,-46.0,-1.0,1.0,-0.002997
4,Angola,Gambia,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.057836,0.011236,0.004739,0,NaN,0.0,2.0,0.006497
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15627,Kosovo,Turkey,0.0,1.0,1.6,0.8,3.0,0.8,2.0,0.960526,0.012821,0.004739,0,NaN,-1.0,1.0,0.008081
15628,Czech Republic,Denmark,2.0,2.0,2.0,0.8,3.4,1.4,2.0,0.960526,0.004739,0.050000,0,NaN,0.0,4.0,-0.045261
15629,Cameroon,China PR,2.0,0.0,1.0,1.2,1.0,1.0,1.0,0.960526,0.022222,0.010638,1,49.0,2.0,2.0,0.011584
15630,Australia,Curaçao,5.0,1.0,0.6,1.2,2.0,0.6,1.0,0.960526,0.037037,0.012195,1,55.0,4.0,6.0,0.024842
